# Restricted Hartree–Fock (RHF) Method

In this tutorial we will make the first step: we will code the restricted HF method.

## Import Section

The only two things we need to import is Psi4 and NumPy. 

In [1]:
import psi4
import numpy as np

Psi4 machinery is needed to specify a molecular system, to select a basis set, and to calculate molecular integrals in it. 
These are the most basic building blocks of any SCF program, and we are going to use them as black-box. We will cover them in detail in other tutorials, but for now we are just fine to use Psi4 software for these purposes.

NumPy is essential for any math-related manipulations. Namely, we will use it for matrix multiplication, matrix diagonalization, etc.

## Molecular Specification

Next, we need to specify a molecular system. It should be something simple, for starters, i.e. an atom.
Let us start with the He atom. It is a convinient "toy system" since there are only two electrons, they occupy a single orbital, and
the system is closed-shell.
For this system, we only need to specify the charge and multiplicity (0, 1).
In Psi4, it is made like this 

In [2]:
mol = psi4.geometry("""
0 1 
He
""")

(a variable's name "mol" will be used for any type of systems).

## Select a Basis Set

The choice of basis set is not a trivial task, one has to have an experience and some level of expertise to make the right choice. 
However, there are plenty of papers on the good practices in quantum chemistry, and DFT specifically, so we will be just following an 
advice from one of those papers. We are going to use basis sets of Ahlrichs and coworkers. Something not too big, just of the right size, like def2-TZVP. 

In [3]:
psi4.set_options({'basis': 'def2-TZVP'})

The set_options function allows tweaking a calculation setup in a lot of various ways. So far we only need to set a basis set for our program. This command basically instructs Psi4 "Go to a folder, where all the basis sets are stored, find a file with the name "def2-TZVP" and read the details about this basis set from there". What will happen if it does not find the file? We will check this in the next section.

## Mints and WFN Objects

Now, when we have specified the molecular system and the basis set that will represent it, it is time to compute the molecular integrals. 
Again, since we are not focusing on it right now, we will use Psi4 functions for that. In Psi4, this is done with Mints and WFN objects.
First, we use WFN to build a wavefunction of our molecule within the chosen basis set

In [4]:
wfn = psi4.core.Wavefunction.build(mol, psi4.core.get_global_option('basis'))

   => Loading Basis Set <=

    Name: DEF2-TZVP
    Role: ORBITAL
    Keyword: BASIS
    atoms 1 entry HE         line    27 file /opt/miniconda3/envs/p4env/share/psi4/basis/def2-tzvp.gbs 



Now we have recieved a message "=> Loading Basis Set <=" and the path to a chosen basis set on your machine. If you type something arbitrary like "'basis': 'def2-TZVPX'" above, you will simply get an error "BasisSetNotFound". But what if I entered a legit basis name and still got an error? Well, this (most likely) means that this basis set is not in your library and you will need to add it there manually. Don't worry, in the basis set tutorial I explain all these things, it is not difficult at all. 

## Molecular Integrals

Now, let's compute molecular integrals to populate the kinetic $\textbf{T}$, electron-nuclear $\textbf{V}$, and electron-electron $\textbf{I}$ tensors. The latter is often called a tensor of electron-repulsion integrals (ERI). 

In [5]:
mints = psi4.core.MintsHelper(wfn.basisset())

When they are computed, we can store them as variables

In [6]:
T  = np.asarray(mints.ao_kinetic())    # Kinetic Energy Integrals
V  = np.asarray(mints.ao_potential())  # Electron-Nuclear Attraction Integrals
I  = np.asarray(mints.ao_eri())        # Electron-Electron Repulsion Integrals

Let's inspect what we've just created. 

In [7]:
print("Dimensions of T: ", T.shape,'\n'
      "Dimensions of V: ", V.shape,'\n'
      "Dimensions of I: ", I.shape) 

Dimensions of T:  (6, 6) 
Dimensions of V:  (6, 6) 
Dimensions of I:  (6, 6, 6, 6)


What is this number 6? That's the number of basis functions we use (go to the basis set file and count manually as an excercise). 
Let's take a closer look at these tensors

In [8]:
print(T)

[[6.85714346 1.43966862 0.22565771 0.         0.         0.        ]
 [1.43966862 1.31120804 0.43093007 0.         0.         0.        ]
 [0.22565771 0.43093007 0.36689846 0.         0.         0.        ]
 [0.         0.         0.         2.5        0.         0.        ]
 [0.         0.         0.         0.         2.5        0.        ]
 [0.         0.         0.         0.         0.         2.5       ]]


What we have got here is a 6 by 6 matrix populated with the values of kinetic integrals. Each integral is given by $$T_{\mu\nu} = \int\phi^*_{\mu}(\textbf{r})\left[-\frac{1}{2}\nabla^2\right]\phi_{\nu}(\textbf{r})d\textbf{r}.$$ As usual, the $\mu$ and $\nu$ values navigate us through this matrix. For example, in this case, $T_{11} = 6.85714346$.

Why is this matrix so weird? Why is here a 3 by 3 block populated with numbers, and another diagonal 3 by 3 block? Well, that's because of the symmetry of the basis functions we use. By inspection, we can find that our basis set consists of three $s$-type and three $p$-type basis functions. As I explain in more detail in the basis set tutorial, the molecular integrals (i.e., the matrix elements) of the basis functions of different symmetries are equal zero. That is exactly what we observe here. All matrix elements of $\textbf{T}$ can be written (here, we assume that the basis functions are real, i.e. we remove an asterix) as $$T_{\mu\nu} = \delta_{\sigma\tau}\int\phi^{\sigma}_{\mu}(\textbf{r})\left[-\frac{1}{2}\nabla^2\right]\phi^{\tau}_{\nu}(\textbf{r})d\textbf{r},\quad\delta_{\sigma\tau}\begin{cases}
    1 & \text{if } \sigma=\tau \\
    0 & \text{if } \sigma\neq\tau.
\end{cases}$$ where $\sigma$ and $\tau$ refer to different symmetries.

Okay, there is an $s$-type and a $p$-type blocks. We have explained the off-diagonal zeros as well $$\begin{bmatrix}
\textbf{S} & \textbf{0} \\
\textbf{0} & \textbf{P} 
\end{bmatrix}.$$ 
But why is the $\textbf{P}$-block diagonal while the $\textbf{S}$-block is not? Try to answer this question.

Let's take a look at the electron-nuclear tensor

In [9]:
print(V)

[[-7.30633341 -3.46504549 -1.5352309   0.          0.          0.        ]
 [-3.46504549 -2.98394096 -1.79402305  0.          0.          0.        ]
 [-1.5352309  -1.79402305 -1.57843741  0.          0.          0.        ]
 [ 0.          0.          0.         -2.12769216  0.          0.        ]
 [ 0.          0.          0.          0.         -2.12769216  0.        ]
 [ 0.          0.          0.          0.          0.         -2.12769216]]


It has a form similar to $\textbf{T}$ but with all elements less than zero. Which also makes absolute sense, since the kinetic energy is positive, while the electron-nuclear attraction potential energy is negative.

Now we can build the core matrix $\textbf{H}^{\text{core}}$ which is a sum of $\textbf{T}$ and $\textbf{V}$:

In [10]:
H = T + V
print(H)

[[-0.44918995 -2.02537687 -1.30957319  0.          0.          0.        ]
 [-2.02537687 -1.67273291 -1.36309298  0.          0.          0.        ]
 [-1.30957319 -1.36309298 -1.21153896  0.          0.          0.        ]
 [ 0.          0.          0.          0.37230784  0.          0.        ]
 [ 0.          0.          0.          0.          0.37230784  0.        ]
 [ 0.          0.          0.          0.          0.          0.37230784]]


This matrix will be used as a source of an initial guess to initiate the SCF cycle. 

## Diagonalization of the Basis Set

Another important matrix to build and analyze is the overlap matrix $\textbf{S}$ with the elements: $$S_{\mu\nu} = \int\phi^*_{\mu}(\textbf{r})\phi_{\nu}(\textbf{r})d\textbf{r}.$$
To call the overlap matrix, we execute: 

In [11]:
S = np.asarray(mints.ao_overlap())
print(S)

[[1.         0.68051026 0.32863185 0.         0.         0.        ]
 [0.68051026 1.         0.75158626 0.         0.         0.        ]
 [0.32863185 0.75158626 1.         0.         0.         0.        ]
 [0.         0.         0.         1.         0.         0.        ]
 [0.         0.         0.         0.         1.         0.        ]
 [0.         0.         0.         0.         0.         1.        ]]


Every diagonal element is $1$ because basis functions of the def2-TZVP basis set are normalized to $1$: $$S_{\mu\mu} = 1.$$ $s$-type basis functions are not orthonormal, because the off-diagonal elements of the $\textbf{S}$-block are non-zero. In contrast, the $\textbf{P}$-block basis functions are orthonormal: $$S^p_{\mu\nu} = \delta_{\mu\nu}.$$ This answers the question on the diagonality of the $\textbf{S}$-block and non-diagonality of the $\textbf{P}$-block in the Molecular Integrals section above. 

To cast our solution in the form of the eigenvalue problem, we need to diagonalize our basis set by diagonalizing the overlap matrix $\textbf{S}$ so it becomes the identity matrix $$\textbf{A}^{\dagger}\textbf{S}\textbf{A} = \textbf{I},$$ and disappears from the matrix equation. The way we use here is to use matrix $\textbf{A}=\textbf{S}^{-1/2}$: $$\textbf{S}^{-1/2}\textbf{S}\textbf{S}^{-1/2} = \textbf{S}^{-1/2}\textbf{S}^{1/2} = \textbf{S}^{0} = \textbf{I},$$ For doing this, we can use the method from the Psi4NumPy tutorial: 

In [12]:
A = mints.ao_overlap()
A.power(-0.5, 1.e-16)
A = np.asarray(A)
print(A)

[[ 1.31662388 -0.64599666  0.15069874  0.          0.          0.        ]
 [-0.64599666  1.85561418 -0.77145695  0.          0.          0.        ]
 [ 0.15069874 -0.77145695  1.43670402  0.          0.          0.        ]
 [ 0.          0.          0.          1.          0.          0.        ]
 [ 0.          0.          0.          0.          1.          0.        ]
 [ 0.          0.          0.          0.          0.          1.        ]]


Now, this is our matrix $\textbf{A}=\textbf{S}^{-1/2}$. Let's apply it to diagonalize $\textbf{S}$:

In [13]:
S_d = A.dot(S).dot(A)
print(np.round(S_d, 17))

[[ 1.0e+00 -5.1e-16  4.1e-16  0.0e+00  0.0e+00  0.0e+00]
 [-4.5e-16  1.0e+00 -2.4e-16  0.0e+00  0.0e+00  0.0e+00]
 [ 4.4e-16 -4.2e-16  1.0e+00  0.0e+00  0.0e+00  0.0e+00]
 [ 0.0e+00  0.0e+00  0.0e+00  1.0e+00  0.0e+00  0.0e+00]
 [ 0.0e+00  0.0e+00  0.0e+00  0.0e+00  1.0e+00  0.0e+00]
 [ 0.0e+00  0.0e+00  0.0e+00  0.0e+00  0.0e+00  1.0e+00]]


We see that the overlap matrix is indeed diagonal now, meaning that all the matrix elements satisfy: $$S_{\mu\nu} = \delta_{\mu\nu}.$$

## Initial Guess

To initiate the SCF procedure an initial guess for the density matrix $\textbf{D}$ has to be made. There are different way of doing this, the one we are using in this tutorial is called the Core guess. The Core initial guess procedure requires diagonalization of the Fock matrix $$\textbf{F} = \textbf{H}^{\text{core}}.$$ The two-electron part of the Fock matrix depends on the density matrix itself $$\textbf{G} = \textbf{G}(\textbf{D}),$$ so to build the Fock operator, one needs to suggest an initial guess $\textbf{D}_0$ and then start the SCF cycle. Therefore, it seems logical to omit the two-electron part at all and diagonalize the Core Hamiltonian $\textbf{H}^{\text{core}}$ to obtain (guess) the initial density matrix $\textbf{D}_0$.

We will use the same matrix $\textbf{A}=\textbf{S}^{-1/2}$ we used for the diagonalization of the overlap matrix $\textbf{S}$ to transform $\textbf{H}^{\text{core}}$ from the original (nonorthonormal) basis set to the orthonormalized one and diagonalize $\textbf{H}^{\text{core}}$ within this basis set:

In [14]:
H_p = A.dot(H).dot(A)
print(H_p)

[[ 1.6867935  -2.86829984 -0.01403363  0.          0.          0.        ]
 [-2.86829984  0.78483247 -0.872163    0.          0.          0.        ]
 [-0.01403363 -0.872163   -0.58103758  0.          0.          0.        ]
 [ 0.          0.          0.          0.37230784  0.          0.        ]
 [ 0.          0.          0.          0.          0.37230784  0.        ]
 [ 0.          0.          0.          0.          0.          0.37230784]]


Compare matrices $\textbf{H}_{\text{core}}$ and $\textbf{H}^{\prime}_{\text{core}}$ in the original and orthonormal basis sets (variables `H` and `H_p`, respectively): they are clearly different. 

Now, we diagonalize `H_p` with `np.linalg.eigh()` function, which arranges eigenvectors $\textbf{C}^{\prime}$ (variable `C_p`) according to their eigenvalues $\{\varepsilon_i\}$ (variable `eps`) from the smallest to the biggest one. Then, transfer these eigenvalues from the orthonormalized basis set to the original one as $\textbf{C} = \textbf{A}\textbf{C}^{\prime}$, and, finally, assemble the density matrix $\textbf{D}$ from the $N/2$ lowest eigenvectors $\textbf{C}$. The number of doubly occupied orbitals $N/2$ is used since we are running the spin-restricted Hartree–Fock method.

In [15]:
eps, C_p = np.linalg.eigh(H_p)                            # Diagonalization of the Core matrix in orthonormal basis set
C = A.dot(C_p)                                            # Transfering eigenvectors from the orthonormal basis set to the original one 
n_el = wfn.nalpha() + wfn.nbeta()                         # Getting the number of electrons N as a sum of alpha and beta electrons
norb = int ( n_el / 2 )                                   # Number of doubly occupied orbitals is N/2 (an integer)
C_occ = C[:, :norb]                                       # Collecting only occupied eigenvectors (according to the Aufbau principle)
D = np.einsum('pi,qi->pq', C_occ, C_occ, optimize=True)   # Assembling the density matrix

Now our initial guess density matrix $\textbf{D}_0$ is ready and we can build the Fock matrix $\textbf{F}$ start the SCF cycle.

## SCF Cycle